# Thêm Thư Viện

In [24]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [25]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2025;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=Library_DWH;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

# Đọc data

## Đọc data từ SQL Server

In [26]:
# Hàm đọc dữ liệu từng phần và xử lý lỗi
def fetch_data_in_batches(query_base, connection, batch_size=100):
    offset = 0
    all_data = []  # Lưu tất cả các hàng hợp lệ
    while True:
        query = f"""
        {query_base}
        ORDER BY So_the
        OFFSET {offset} ROWS FETCH NEXT {batch_size} ROWS ONLY
        """
        try:
            # Đọc dữ liệu batch hiện tại
            df_batch = pd.read_sql(query, connection)
            if df_batch.empty:  # Nếu không còn dữ liệu, dừng vòng lặp
                break
            all_data.append(df_batch)  # Lưu batch hợp lệ
            offset += batch_size  # Tăng offset để đọc batch tiếp theo
        except Exception as e:
            print(f"Lỗi xảy ra khi xử lý batch từ {offset}: {e}")
            offset += batch_size  # Bỏ qua batch bị lỗi và tiếp tục
    # Gộp tất cả các batch thành DataFrame duy nhất
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

In [ ]:
query_Bandoc = """
SELECT dbo.DecodeUTF8String(So_the) AS So_the,
       dbo.DecodeUTF8String(Ho_ten) AS Ho_ten,
       Ngay_sinh,
       Dan_toc_ID,
       Trinh_do_ID,
       dbo.DecodeUTF8String(So_dien_thoai) AS So_dien_thoai,
       dbo.DecodeUTF8String(Nghe_nghiep) AS Nghe_nghiep,
       dbo.DecodeUTF8String(Co_quan) AS Co_quan,
       dbo.DecodeUTF8String(chuc_vu) AS chuc_vu,
       dbo.DecodeUTF8String(Dia_chi) AS Dia_chi,
       dbo.DecodeUTF8String(Dia_chi_thuong_tru) AS Dia_chi_thuong_tru,
       dbo.DecodeUTF8String(Khoa_hoc) AS Khoa_hoc,
       Lop,
       Anh,
       Ngay_cap,
       Ngay_het_han,
       Email,
       Nhom_ID,
       Nhom_nghanh_nghe_ID,
       Gioi_tinh,
       Status,
       dbo.DecodeUTF8String(Ghi_chu) AS Ghi_chu,
       Mat_khau
FROM Ban_doc
"""
df_bandoc = fetch_data_in_batches(query_Bandoc, conn_libol, batch_size=100) # Gọi hàm để lấy dữ liệu
print(df_bandoc) # Hiển thị kết quả

C:\Users\admin\AppData\Local\Temp\ipykernel_29572\705386798.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_batch = pd.read_sql(query, connection)


        So_the            Ho_ten  Ngay_sinh  Dan_toc_ID  Trinh_do_ID  \
0     00105024   NGUYỄN TẤN GIÁO 1982-08-10           1            3   
1     01101044     VŨ NGỌC HOÀNG 1983-08-05           1            9   
2     01101056   NGUYỄN VĂN KHÔI        NaT           1            9   
3     01101062     PHAN HỮU LÀNH        NaT           1            9   
4     01101077       VÕ TUẤN NAM        NaT           1            9   
..         ...               ...        ...         ...          ...   
115   01710084         HỒ VĂN VĨ        NaT           1            9   
116   02404093         LƯ TỐ NHƯ        NaT           1            9   
117   A1404030  NG. MAI THANH HÀ        NaT           1            9   
118  ÊN1102042   NGUYỄN PHÚ HIỀN        NaT           1            9   
119   O1101004    NGUYỄN VĂN BẢO 1983-04-21           1            3   

    So_dien_thoai Nghe_nghiep        Co_quan chuc_vu  \
0                        None  Trường ĐHSPKT    None   
1                      

# Xử lý data

## Thêm 1 dòng giả định none

In [28]:
# Tạo DataFrame `new_row` chứa dòng dữ liệu giả định
new_row = pd.DataFrame({
    'So_the': ['0'],
    'Ho_ten': ['(Không xác định)'],
    'Ngay_sinh': ['1024-01-01 00:00:00'],
    'ID_nien_khoa': [0],
    'ID_dan_toc': [56],
    'ID_trinh_do': [0],
    'ID_lop': [0],
    'Ngay_cap': ['1024-01-01 00:00:00'],
    'Ngay_het_han': ['1024-01-01 00:00:00'],
    'ID_nhom_ban_doc': [0],
    'ID_nhom_nghanh_nghe': [0],
})

# Thêm dòng dữ liệu giả định vào `df` bằng `pd.concat`
df_bandoc = pd.concat([df_bandoc, new_row], ignore_index=True) # Thêm vào dataframe
df_bandoc['So_the'] = df_bandoc['So_the'].astype(str)
df_bandoc = df_bandoc.sort_values(by="So_the", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn           
print(df_bandoc)

        So_the            Ho_ten            Ngay_sinh  Dan_toc_ID  \
0            0  (Không xác định)  1024-01-01 00:00:00         NaN   
1     00105024   NGUYỄN TẤN GIÁO  1982-08-10 00:00:00         1.0   
2     01101044     VŨ NGỌC HOÀNG  1983-08-05 00:00:00         1.0   
3     01101056   NGUYỄN VĂN KHÔI                  NaT         1.0   
4     01101062     PHAN HỮU LÀNH                  NaT         1.0   
..         ...               ...                  ...         ...   
116   01710084         HỒ VĂN VĨ                  NaT         1.0   
117   02404093         LƯ TỐ NHƯ                  NaT         1.0   
118   A1404030  NG. MAI THANH HÀ                  NaT         1.0   
119   O1101004    NGUYỄN VĂN BẢO  1983-04-21 00:00:00         1.0   
120  ÊN1102042   NGUYỄN PHÚ HIỀN                  NaT         1.0   

     Trinh_do_ID So_dien_thoai Nghe_nghiep        Co_quan chuc_vu  \
0            NaN           NaN         NaN            NaN     NaN   
1            3.0                 

## Xử lý data rỗng hoặc " "

In [29]:
df_bandoc = df_bandoc.replace(np.nan, None)
df_bandoc = df_bandoc.replace('', None)
print(df_bandoc[['Nghe_nghiep', 'Co_quan', 'chuc_vu']])

    Nghe_nghiep        Co_quan chuc_vu
0          None           None    None
1          None  Trường ĐHSPKT    None
2          None  Trường ĐHSPKT    None
3          None         ĐHSPKT    None
4          None  Trường ĐHSPKT    None
..          ...            ...     ...
116        None         ĐHSPKT    None
117        None         ĐHSPKT    None
118        None         ĐHSPKT    None
119        None         ĐHSPKT    None
120        None         ĐHSPKT    None

[121 rows x 3 columns]


## Xử lý kiểu date

In [30]:
query_date = "SELECT Date_key FROM olap.DIM_date"
df_date = pd.read_sql(query_date, conn_dwh_library)

date_ids = set(df_date['Date_key'])

# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong date_key của bảng DIM_date hay không ?
df_bandoc['Ngay_sinh'] = pd.to_datetime(df_bandoc['Ngay_sinh'], errors='coerce')
df_bandoc['Ngay_cap'] = pd.to_datetime(df_bandoc['Ngay_cap'], errors='coerce')
df_bandoc['Ngay_het_han'] = pd.to_datetime(df_bandoc['Ngay_het_han'], errors='coerce')

df_bandoc['Ngay_sinh'] = df_bandoc['Ngay_sinh'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)
df_bandoc['Ngay_cap'] = df_bandoc['Ngay_cap'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)
df_bandoc['Ngay_het_han'] = df_bandoc['Ngay_het_han'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)

print(df_bandoc[['Ngay_sinh', 'Ngay_cap', 'Ngay_het_han']])

     Ngay_sinh  Ngay_cap  Ngay_het_han
0            0         0             0
1     19820810  20051012      20060830
2     19830805  20021023      20050830
3            0  20021023      20040831
4            0  20020925      20050910
..         ...       ...           ...
116          0  20021008      20040831
117          0  20031027      20040831
118          0  20031007      20040831
119   19830421  20031201      20040831
120          0  20030924      20040831

[121 rows x 3 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_29572\3773408894.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_date = pd.read_sql(query_date, conn_dwh_library)
C:\Users\admin\AppData\Local\Temp\ipykernel_29572\3773408894.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_bandoc['Ngay_sinh'] = pd.to_datetime(df_bandoc['Ngay_sinh'], errors='coerce')
C:\Users\admin\AppData\Local\Temp\ipykernel_29572\3773408894.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_bandoc['Ngay_cap'] = pd.to_datetime(df_bandoc['Ngay_cap'], errors='coerce')
C:\Users\adm

## Xử lý Nien_khoa

In [31]:
df_bandoc['Khoa_hoc'] = df_bandoc['Khoa_hoc'].str.replace(r'\s*-\s*', '-', regex=True)
df_bandoc['Khoa_hoc'] = df_bandoc['Khoa_hoc'].replace('', np.nan)
df_bandoc = df_bandoc.dropna(subset=['Khoa_hoc'])

query_Nienkhoa = "SELECT ID_nien_khoa, Ten_nien_khoa FROM olap.DIM_Nien_khoa"
df_nienkhoa = pd.read_sql(query_Nienkhoa, conn_dwh_library)

# Tạo một từ điển ánh xạ giữa Ten_nien_khoa và ID_nien_khoa
mapping = df_nienkhoa.set_index('Ten_nien_khoa')['ID_nien_khoa'].to_dict()
df_bandoc['ID_nien_khoa'] = df_bandoc['Khoa_hoc'].map(mapping)

print(df_bandoc['ID_nien_khoa'])

1      136
2      136
3      136
4      136
5      136
      ... 
116    136
117    136
118    136
119    136
120    136
Name: ID_nien_khoa, Length: 120, dtype: int64


C:\Users\admin\AppData\Local\Temp\ipykernel_29572\3916828476.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nienkhoa = pd.read_sql(query_Nienkhoa, conn_dwh_library)


## Xử lý Dan_toc

In [32]:
# đọc dữ liệu lấy từ bộ về
df_Data_Dim_Dan_toc = pd.read_csv("./data_dan_toc.csv")
df_Data_Dim_Dan_toc = df_Data_Dim_Dan_toc.where(pd.notnull(df_Data_Dim_Dan_toc), None)
# đọc dữ liệu đã lưu trong sql server
query_Dan_toc = "SELECT Id, dbo.DecodeUTF8String(Dan_toc) AS Dan_toc FROM Dan_toc "
df_Dan_toc = pd.read_sql(query_Dan_toc, conn_libol)

#tìm và mapping 2 bảng lại
df_mapping = df_Dan_toc.copy()
df_mapping['Mapping Mã'] = None
df_mapping['CSV Mã'] = df_Data_Dim_Dan_toc['Mã']
df_mapping['CSV Tên'] = df_Data_Dim_Dan_toc['Tên']
df_mapping['CSV Tên khác'] = df_Data_Dim_Dan_toc['Tên gọi khác'].str.lower()
for i, dan_toc in enumerate(df_mapping['Dan_toc']):
    dan_toc = dan_toc.lower()
    dan_toc_bogach = dan_toc.replace("-", " ")
    dan_toc_botrong = dan_toc.replace(" ", "-")

    for j, row in df_mapping.iterrows():
        CSV_ten = row['CSV Tên'].lower() if pd.notna(row['CSV Tên']) else ""
        CSV_ten_khac = row['CSV Tên khác'].lower() if pd.notna(row['CSV Tên khác']) else ""
        if ((pd.notna(CSV_ten) and dan_toc == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc in CSV_ten_khac) or 
            (pd.notna(CSV_ten) and dan_toc_bogach == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc_bogach in CSV_ten_khac) or
            (pd.notna(CSV_ten) and dan_toc_botrong == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc_botrong in CSV_ten_khac)):
            df_mapping.at[i, 'Mapping Mã'] = row['CSV Mã']
            break
        else:
            df_mapping.at[i, 'Mapping Mã'] = 56
print(df_mapping)

    Id  Dan_toc Mapping Mã  CSV Mã CSV Tên  \
0    1     Kinh        1.0     1.0    Kinh   
1    2    Mường        3.0     2.0     Tày   
2    3      Tày        2.0     3.0    Thái   
3    4     Thái        3.0     4.0     Hoa   
4    5      Hoa        4.0     5.0  Khơ-me   
..  ..      ...        ...     ...     ...   
68  71     Ê Đê       12.0     NaN     NaN   
69  72      Thổ        2.0     NaN     NaN   
70  73    Kờ Ho         56     NaN     NaN   
71  74     Jrai         56     NaN     NaN   
72  75  Châu mạ       28.0     NaN     NaN   

                                         CSV Tên khác  
0                                                việt  
1           thổ, ngạn, phén, thù lao, pa dí, tày khao  
2   tày đăm, tày mười, tày thanh, mán thanh, hàng ...  
3   hán, triều châu, phúc kiến, quảng đông, hải na...  
4              cur, cul, cu, thổ, việt gốc miên, krôm  
..                                                ...  
68                                                NaN  

C:\Users\admin\AppData\Local\Temp\ipykernel_29572\2376671368.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_Dan_toc = pd.read_sql(query_Dan_toc, conn_libol)


In [33]:
map_dict = dict(zip(df_mapping['Id'], df_mapping['Mapping Mã'])) # Tạo map_dict để ánh xạ từ Id sang Mapping_Ma trong df_mapping

for index, row in df_bandoc.iterrows(): # Lặp qua từng dòng trong df_ban_doc để cập nhật Dan_toc_ID
    
    if not pd.isna(row['Dan_toc_ID']): # Kiểm tra nếu Dan_toc_ID rỗng (None hoặc NaN)
        if row['Dan_toc_ID'] in map_dict: # Kiểm tra nếu Dan_toc_ID có trong map_dict
            df_bandoc.at[index, 'Dan_toc_ID'] = map_dict[row['Dan_toc_ID']]# Nếu tìm thấy, thay thế bằng giá trị Mapping_Ma
        else:
            df_bandoc.at[index, 'Dan_toc_ID'] = 56 # Nếu không tìm thấy, gán Dan_toc_ID bằng 0
    else:
        df_bandoc.at[index, 'Dan_toc_ID'] = 56
print(df_bandoc['Dan_toc_ID'])

1      1.0
2      1.0
3      1.0
4      1.0
5      1.0
      ... 
116    1.0
117    1.0
118    1.0
119    1.0
120    1.0
Name: Dan_toc_ID, Length: 120, dtype: object


## Xử lý Trinh_do

In [34]:
query_Trinhdo = "SELECT ID_trinh_do FROM olap.DIM_Trinh_do"
df_trinhdo = pd.read_sql(query_Trinhdo, conn_dwh_library)

trinhdo_ids = set(df_trinhdo['ID_trinh_do'])

# Duyệt qua từng giá trị Lop trong df_bandoc và kiểm tra xem nó có nằm trong trinhdo_ids hay không. 
# Nếu có thì dữ nguyên nếu không, gán giá trị bằng 0.
df_bandoc['Trinh_do_ID'] = df_bandoc['Trinh_do_ID'].apply(lambda x: x if x in trinhdo_ids else 0)

print(df_bandoc['Trinh_do_ID'])

1      3.0
2      9.0
3      9.0
4      9.0
5      9.0
      ... 
116    9.0
117    9.0
118    9.0
119    3.0
120    9.0
Name: Trinh_do_ID, Length: 120, dtype: float64


C:\Users\admin\AppData\Local\Temp\ipykernel_29572\1481478981.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_trinhdo = pd.read_sql(query_Trinhdo, conn_dwh_library)


## Xử lý Lop

In [35]:
query_lop = "SELECT ID_lop FROM olap.DIM_Lop"
df_lop = pd.read_sql(query_lop, conn_dwh_library)

lop_ids = set(df_lop['ID_lop'])

# Duyệt qua từng giá trị Lop trong df_bandoc và kiểm tra xem nó có nằm trong lop_ids hay không. 
# Nếu có, chuyển sang dạng chữ in hoa (upper()), nếu không, gán giá trị bằng 0.
df_bandoc['Lop'] = df_bandoc['Lop'].str.upper()
df_bandoc['Lop'] = df_bandoc['Lop'].apply(lambda x: x.upper() if x in lop_ids else 0)

print(df_bandoc['Lop'])

C:\Users\admin\AppData\Local\Temp\ipykernel_29572\529711181.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lop = pd.read_sql(query_lop, conn_dwh_library)


1      011051A
2       011011
3       011011
4       011011
5       011011
        ...   
116          0
117    01404CA
118    01404TV
119     011011
120          0
Name: Lop, Length: 120, dtype: object


## Xử lý Nhom_ban_doc

In [36]:
query_Nhombandoc = "SELECT ID_nhom_ban_doc FROM olap.DIM_Nhom_ban_doc"
df_nhombandoc = pd.read_sql(query_Nhombandoc, conn_dwh_library)

nhombandoc_ids = set(df_nhombandoc['ID_nhom_ban_doc'])

# Duyệt qua từng giá trị Lop trong df_bandoc và kiểm tra xem nó có nằm trong nhombandoc_ids hay không. 
# Nếu có thì dữ nguyên nếu không, gán giá trị bằng 0.
df_bandoc['Nhom_ID'] = df_bandoc['Nhom_ID'].apply(lambda x: x if x in nhombandoc_ids else 0)

print(df_bandoc['Nhom_ID'])

1      9.0
2      9.0
3      9.0
4      9.0
5      9.0
      ... 
116    9.0
117    9.0
118    9.0
119    9.0
120    9.0
Name: Nhom_ID, Length: 120, dtype: float64


C:\Users\admin\AppData\Local\Temp\ipykernel_29572\3965989656.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nhombandoc = pd.read_sql(query_Nhombandoc, conn_dwh_library)


## Xử lý Nhom_nghanh_nghe

In [37]:
query_Nhomnghanhnghe = "SELECT ID_nhom_nghanh_nghe FROM olap.DIM_Nhom_nghanh_nghe"
df_nhomnghanhnge = pd.read_sql(query_Nhomnghanhnghe, conn_dwh_library)

nhomnghanhnghe_ids = set(df_nhomnghanhnge['ID_nhom_nghanh_nghe'])

# Duyệt qua từng giá trị Lop trong df_bandoc và kiểm tra xem nó có nằm trong nhomnghanhnghe_ids hay không. 
# Nếu có thì dữ nguyên nếu không, gán giá trị bằng 0.
df_bandoc['Nhom_nghanh_nghe_ID'] = df_bandoc['Nhom_nghanh_nghe_ID'].apply(lambda x: x if x in nhomnghanhnghe_ids else 0)

print(df_bandoc['Nhom_nghanh_nghe_ID'])

1       98.0
2       94.0
3       94.0
4       65.0
5       65.0
       ...  
116      0.0
117    128.0
118     63.0
119     94.0
120     64.0
Name: Nhom_nghanh_nghe_ID, Length: 120, dtype: float64


C:\Users\admin\AppData\Local\Temp\ipykernel_29572\938135672.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nhomnghanhnge = pd.read_sql(query_Nhomnghanhnghe, conn_dwh_library)


## Xử lý duplicate cho primary key

In [38]:
df_bandoc = df_bandoc.drop_duplicates(subset='So_the').reset_index(drop=True) 
print(df_bandoc)

        So_the            Ho_ten  Ngay_sinh Dan_toc_ID  Trinh_do_ID  \
0     00105024   NGUYỄN TẤN GIÁO   19820810        1.0          3.0   
1     01101044     VŨ NGỌC HOÀNG   19830805        1.0          9.0   
2     01101056   NGUYỄN VĂN KHÔI          0        1.0          9.0   
3     01101062     PHAN HỮU LÀNH          0        1.0          9.0   
4     01101077       VÕ TUẤN NAM          0        1.0          9.0   
..         ...               ...        ...        ...          ...   
115   01710084         HỒ VĂN VĨ          0        1.0          9.0   
116   02404093         LƯ TỐ NHƯ          0        1.0          9.0   
117   A1404030  NG. MAI THANH HÀ          0        1.0          9.0   
118   O1101004    NGUYỄN VĂN BẢO   19830421        1.0          3.0   
119  ÊN1102042   NGUYỄN PHÚ HIỀN          0        1.0          9.0   

    So_dien_thoai Nghe_nghiep        Co_quan chuc_vu  \
0            None        None  Trường ĐHSPKT    None   
1            None        None  Trườ

## Load data

### [Nếu cần] Clear bảng

In [39]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Ban_doc"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [22]:
# Tạo cursor để thao tác với cơ sở dữ liệu
cursor_dwh = conn_dwh_library.cursor()

# Chuẩn bị câu lệnh chèn dữ liệu
insert_query = """
                INSERT INTO olap.DIM_Ban_doc (
                    ID_ban_doc, Ho_ten, Ngay_sinh, 
                    ID_nien_khoa, ID_dan_toc, ID_trinh_do,
                    So_dien_thoai, Nghe_nghiep, Co_quan, Chuc_vu,
                    Dia_chi_tam_tru, Dia_chi_thuong_tru, ID_lop,
                    Anh, Ngay_cap, Ngay_het_han, Email, ID_nhom_ban_doc,
                    ID_nhom_nghanh_nghe, Gioi_tinh, Tinh_trang, Ghi_chu, Mat_khau
                ) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
               """

# Chuyển đổi dữ liệu từ DataFrame thành danh sách các tuple để chèn
data_to_insert = [
    (
        row['So_the'], row['Ho_ten'], row['Ngay_sinh'], 
        row['ID_nien_khoa'], row['Dan_toc_ID'], row['Trinh_do_ID'],
        row['So_dien_thoai'], row['Nghe_nghiep'], row['Co_quan'], row['chuc_vu'],
        row['Dia_chi'], row['Dia_chi_thuong_tru'], row['Lop'],
        row['Anh'], row['Ngay_cap'], row['Ngay_het_han'], row['Email'], row['Nhom_ID'],
        row['Nhom_nghanh_nghe_ID'], row['Gioi_tinh'], row['Status'], row['Ghi_chu'], row['Mat_khau']
    )
    for index, row in df_bandoc.iterrows()
]

cursor_dwh.executemany(insert_query, data_to_insert) # Sử dụng executemany để chèn dữ liệu cùng lúc
conn_dwh_library.commit() # Commit thay đổi
cursor_dwh.close() # Đóng cursor và kết nối
conn_dwh_library.close()